# Proposed validation — review before it runs

**What this measures:** `dice_gap` (segformer3d_dice − segresnet_dice) measures whether the real `SegFormer3D` module, trained from scratch under an identical fixed-budget protocol as `SegResNet`, reaches segmentation-accuracy parity with an established MONAI net on the real, cached Task01_BrainTumour 8/4 subset fetched once via `monai.apps.DecathlonDataset`.

**Target metric:** `dice_gap`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_segformer3d_brats_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the script at `eval/eval_segformer3d_brats_parity.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_segformer3d_brats_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
#!/usr/bin/env python
"""
eval/eval_segformer3d_brats_parity.py

Fixed-budget, from-scratch Dice-parity check between SegFormer3D (added by this PR)
and MONAI's existing SegResNet, trained on a fixed real 8/4 subset of the cached
Task01_BrainTumour dataset. Both models are always trained/scored from whichever
arm is running this script (no two-arm delta is used for the target, per
held_constant/avoid in validation.yaml); on the `dev` baseline SegFormer3D cannot be
imported at all, so its figures degrade to zero and dice_gap reads against the fixed
-0.05 bound instead of comparing across arms.

Fixes applied per reviewer feedback ("fix the issues in the test and run again"):
  * deterministic subset selection -- the 8/4 subset is chosen from a list SORTED by
    image path, not by whatever order DecathlonDataset happens to enumerate files in.
  * a stable, working-directory cache path so the (~7GB) Task01_BrainTumour download
    is fetched once and reused by every subsequent invocation instead of a fresh
    temp dir per run.
"""
from __future__ import annotations

In [ ]:
import argparse
import itertools
import json
import os
import sys
import time

In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.insert(0, REPO_ROOT)

import torch  # noqa: E402

parser = argparse.ArgumentParser()
parser.add_argument("--variant", default=None)
parser.add_argument("--ref", default=None)
parser.add_argument("--seed", default=None)
_args, _unknown = parser.parse_known_args()

In [ ]:
SEED = 42
SMOKE = os.environ.get("REMYX_SMOKE") == "1"
N_TRAIN, N_VAL, N_ITERS, CROP = (1, 1, 2, (16, 16, 16)) if SMOKE else (8, 4, 200, (64, 64, 64))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = os.path.join(os.getcwd(), "brats_cache")  # stable path -> reused across runs

from monai.utils import set_determinism  # noqa: E402

In [ ]:
set_determinism(seed=SEED)

# ---- defensive import of EXACTLY the symbol this diff adds -----------------------
try:
    from monai.networks.nets import SegFormer3D
except (ImportError, AttributeError):
    SegFormer3D = None

In [ ]:
# ---- guardrail: existing shared nets still import from the edited __init__.py ----
_CORE_NETS = ["UNet", "SegResNet", "BasicUNet", "DynUNet", "VNet", "AttentionUnet"]
_ok = 0
for _name in _CORE_NETS:
    try:
        import monai.networks.nets as _nets_mod

        getattr(_nets_mod, _name)
        _ok += 1
    except (ImportError, AttributeError):
        pass
existing_nets_import_success_rate = _ok / len(_CORE_NETS)

In [ ]:
# ---- real data: cached Task01_BrainTumour, fixed deterministic 8/4 subset --------
from monai.apps import DecathlonDataset  # noqa: E402
from monai.data import Dataset as MonaiDataset, DataLoader  # noqa: E402
from monai.losses import DiceLoss  # noqa: E402
from monai.metrics import DiceMetric  # noqa: E402
from monai.networks.nets import SegResNet  # noqa: E402
from monai.transforms import (  # noqa: E402
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ConvertToMultiChannelBasedOnBratsClassesd,
    SpatialPadd,
    RandSpatialCropd,
    CenterSpatialCropd,
    NormalizeIntensityd,
    EnsureTyped,
)

In [ ]:
base_tf = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    SpatialPadd(keys=["image", "label"], spatial_size=CROP),
]
train_transform = Compose(base_tf + [RandSpatialCropd(keys=["image", "label"], roi_size=CROP, random_size=False),
                                      NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
                                      EnsureTyped(keys=["image", "label"])])
val_transform = Compose(base_tf + [CenterSpatialCropd(keys=["image", "label"], roi_size=CROP),
                                    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
                                    EnsureTyped(keys=["image", "label"])])

In [ ]:
_train_full = DecathlonDataset(root_dir=CACHE_DIR, task="Task01_BrainTumour", section="training",
                                transform=None, download=True, cache_rate=0.0, val_frac=0.2, seed=SEED)
_val_full = DecathlonDataset(root_dir=CACHE_DIR, task="Task01_BrainTumour", section="validation",
                              transform=None, download=True, cache_rate=0.0, val_frac=0.2, seed=SEED)

# deterministic fixed subset: sort by image path, then slice -- fixes prior nondeterminism
_train_items = sorted(_train_full.data, key=lambda d: d["image"])[:N_TRAIN]
_val_items = sorted(_val_full.data, key=lambda d: d["image"])[:N_VAL]

In [ ]:
train_ds = MonaiDataset(data=_train_items, transform=train_transform)
val_ds = MonaiDataset(data=_val_items, transform=val_transform)
train_loader = DataLoader(train_ds, batch_size=min(2, N_TRAIN), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

def run_arm(model: torch.nn.Module) -> tuple[float, int, float, float]:
    """Train `model` from scratch for N_ITERS steps, then score Dice on the fixed val subset."""
    model = model.to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = DiceLoss(sigmoid=True, squared_pred=True)

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    iter_times: list[float] = []
    model.train()
    stream = itertools.cycle(train_loader)
    for step in range(N_ITERS):
        batch = next(stream)
        images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        if step >= min(2, max(N_ITERS - 1, 0)):  # skip warm-up iteration(s)
            iter_times.append(time.perf_counter() - t0)

    median_iter_s = sorted(iter_times)[len(iter_times) // 2] if iter_times else 0.0
    train_time_s = median_iter_s * N_ITERS
    max_mem_mb = torch.cuda.max_memory_allocated(DEVICE) / 1e6 if DEVICE == "cuda" else 0.0

    dice_metric = DiceMetric(include_background=True, reduction="mean")
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            pred = (torch.sigmoid(model(images)) > 0.5).float()
            dice_metric(y_pred=pred, y=labels)
    dice = float(dice_metric.aggregate().item())
    dice_metric.reset()
    return dice, int(params), float(train_time_s), float(max_mem_mb)

In [ ]:
segresnet = SegResNet(spatial_dims=3, in_channels=4, out_channels=3)
segresnet_dice, segresnet_params, segresnet_train_time_s, segresnet_max_mem_mb = run_arm(segresnet)

if SegFormer3D is not None:
    segformer3d = SegFormer3D(in_channels=4, out_channels=3)
    segformer3d_dice, segformer3d_params, segformer3d_train_time_s, segformer3d_max_mem_mb = run_arm(segformer3d)
else:
    # baseline (`dev`) fallback: SegFormer3D does not exist pre-change -> degraded/zeroed metrics
    segformer3d_dice, segformer3d_params, segformer3d_train_time_s, segformer3d_max_mem_mb = 0.0, 0, 0.0, 0.0

In [ ]:
dice_gap = segformer3d_dice - segresnet_dice

print(json.dumps({
    "dice_gap": dice_gap,
    "existing_nets_import_success_rate": existing_nets_import_success_rate,
    "segformer3d_dice": segformer3d_dice,
    "segresnet_dice": segresnet_dice,
    "segformer3d_params": segformer3d_params,
    "segresnet_params": segresnet_params,
    "segformer3d_train_time_s": segformer3d_train_time_s,
    "segresnet_train_time_s": segresnet_train_time_s,
    "segformer3d_max_mem_mb": segformer3d_max_mem_mb,
    "segresnet_max_mem_mb": segresnet_max_mem_mb,
}))
sys.exit(0)

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: segformer3d-brats-architecture-parity
    suite: "eval/eval_segformer3d_brats_parity.py"
    scorer: dice_gap
    baseline: dev
    metrics:
      - name: dice_gap
        role: target
        direction: max
        threshold: -0.05
      - name: existing_nets_import_success_rate
        role: guardrail
        direction: max
        threshold: 1.0
      - name: segformer3d_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segresnet_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segformer3d_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segresnet_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segformer3d_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segresnet_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segformer3d_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
      - name: segresnet_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
    policy: {guardrail_veto: true}
held_constant:
  - "same fixed 8/4 subset of the real Task01_BrainTumour training list: deterministically selected with one fixed seed for both models, identical inputs for both arms"
  - "same crop geometry: (64, 64, 64) voxels, same seed, same crop/resize pipeline applied to the real Task01_BrainTumour volumes for both models"
  - "same optimizer, learning rate and loss: Adam(lr=1e-4) with DiceLoss(sigmoid=True, squared_pred=True), identical for both models"
  - "same fixed step budget: 200 from-scratch training iterations for both models, no pretrained weights for either"
  - "same torch/numpy/monai seed across both models via monai.utils.set_determinism, single seed only, no multi-seed averaging"
  - "Task01_BrainTumour is fetched exactly once via monai.apps.DecathlonDataset(root_dir=<working dir>, task='Task01_BrainTumour', download=True) and cached in the working directory; repeat runs reuse the cache rather than re-downloading"
avoid:
  - "this is a fixed-budget, from-scratch comparison on a fixed real 8/4 Task01_BrainTumour subset -- a parity signal, not a reproduction of either paper's full training protocol or fully converged accuracy on all of BraTS"
  - "the multi-gigabyte Task01_BrainTumour download is now exercised per user_guidance; it is fetched once into the working directory and cached, so repeat invocations of this script must detect the existing cache and skip re-downloading to stay practical"
  - "no two-arm delta is used for the target: the dev baseline cannot import SegFormer3D at all, so both models are trained and scored from the feature arm and the gap is read against a fixed bound"
  - "train time, parameter counts and peak device memory are reported only as cost references, never as a pass/fail gate on model quality"
  - "no invented paper Dice numbers: only Dice computed by this script's own from-scratch training run on the real cached subset is reported"
  - "single seed only, as instructed: no seed-to-seed variance estimate exists for dice_gap, so the fixed -0.05 tolerance band remains a fixed bound rather than a noise-derived one"
compute:
  tier: gpu
  timeout_s: 5400
provenance:
  dice_gap: "user_guidance (fixed-budget from-scratch Dice comparison on the real Task01_BrainTumour 8/4 subset, no two-arm delta)"
  existing_nets_import_success_rate: "inferred -- regression guardrail for the monai/networks/nets/__init__.py edit this PR makes, since that file is shared by every existing net"
  segformer3d_dice: "user_guidance (mean validation Dice per model)"
  segresnet_dice: "user_guidance (mean validation Dice per model)"
  segformer3d_params: "user_guidance (params reported as a cost reference only)"
  segresnet_params: "user_guidance (params reported as a cost reference only)"
  segformer3d_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segresnet_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segformer3d_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  segresnet_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  held_constant: "user_guidance (same fixed 8/4 subset and seed, downloaded once via monai.apps.DecathlonDataset) refined by protocol_doc:monai/networks/nets/segformer3d.py for the channel/crop geometry"
  compute: "inferred -- timeout_s raised from 3600 to 5400 to cover the one-time Task01_BrainTumour download alongside the unchanged 200-iteration x2-model training budget"
  suite: "synthesized (R1 maturity repo: tests + CI only, no BraTS benchmark harness exists to run (a) against); loads the real cached Task01_BrainTumour subset via monai.apps.DecathlonDataset per user_guidance instead of synthetic volumes; execution bugs from the prior attempt (nondeterministic subset selection, a re-download-on-every-run caching bug) are fixed in the script per user_guidance 'fix the issues in the test and run again'"
```